<a href="https://colab.research.google.com/github/mayankjpt-svg/AutoAI/blob/main/gpt2_sft_children_stories_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GPT-2 Fine-Tuning on SFT / Text Datasets

This Colab notebook:

1. Loads a Hugging Face dataset, default: `deven367/babylm-100M-children-stories`.
2. Cleans and formats examples as causal-LM training text.
3. Runs a fixed prompt eval set **before** fine-tuning.
4. Fine-tunes GPT-2 using Hugging Face `Trainer`.
5. Runs the **same** prompt eval set after fine-tuning.
6. Saves the model, tokenizer, baseline generations, fine-tuned generations, and a comparison file.

Default mode is plain causal language modeling because the children-stories dataset is story/text data, not instruction-response data.  
The notebook also supports prompt-response SFT datasets by setting prompt/response columns in the config cell.

## 0. Runtime

Use a GPU runtime:

`Runtime → Change runtime type → T4 GPU or better`

For free Colab, start with `distilgpt2` or reduce `MAX_TRAIN_EXAMPLES`.  
For better quality, use `gpt2` and train longer.

In [ ]:
# @title Install dependencies
!pip -q install -U "transformers>=4.44.0" "datasets>=2.20.0" "accelerate>=0.33.0" "evaluate>=0.4.2" "huggingface_hub>=0.24.0" "safetensors>=0.4.3"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 63.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 693.4/693.4 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 12.7 MB/s eta 0:00:00


In [ ]:
# @title Imports and environment
import os
import re
import json
import math
import time
import random
import inspect
from pathlib import Path
from typing import Dict, List, Any, Optional

import torch
from datasets import load_dataset, Dataset, DatasetDict
from huggingface_hub import login

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer,
    set_seed,
)

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_DISABLED"] = "true"

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Torch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [ ]:
# @title Config
from google.colab import userdata
from google.colab import drive

# Hugging Face auth
# Add this in Colab: left sidebar → 🔑 Secrets → HF_TOKEN
HF_TOKEN = userdata.get("HF_TOKEN") or userdata.get("HUGGINGFACE_TOKEN")
if HF_TOKEN:
    login(token=HF_TOKEN)
else:
    print("No HF_TOKEN found. Public datasets/models will still work.")

# Storage
MOUNT_DRIVE = True # @param {type:"boolean"}
PROJECT_DIR = "/content/drive/MyDrive/gpt2_children_stories_ft" # @param {type:"string"}

if MOUNT_DRIVE:
    drive.mount("/content/drive")
Path(PROJECT_DIR).mkdir(parents=True, exist_ok=True)

# Dataset
DATASET_NAME = "deven367/babylm-100M-children-stories" # @param {type:"string"}
DATASET_CONFIG = "" # @param {type:"string"}
DATASET_SPLIT = "train" # @param {type:"string"}

# Model
MODEL_NAME = "gpt2" # @param ["gpt2", "distilgpt2"] {allow-input: true}
OUTPUT_DIR = f"{PROJECT_DIR}/runs/{MODEL_NAME.replace('/', '_')}_{DATASET_NAME.replace('/', '_')}"
FINAL_MODEL_DIR = f"{OUTPUT_DIR}/final_model"

# Data mode
# auto = raw text if no prompt/response columns are set; prompt_response = instruction-style SFT
DATASET_MODE = "auto" # @param ["auto", "raw_text", "prompt_response"]
TEXT_COLUMN = "" # @param {type:"string"}
PROMPT_COLUMN = "" # @param {type:"string"}
RESPONSE_COLUMN = "" # @param {type:"string"}
SYSTEM_PROMPT = "" # @param {type:"string"}

# Training size / speed
MAX_TRAIN_EXAMPLES = 20000 # @param {type:"integer"}
MAX_EVAL_EXAMPLES = 1000 # @param {type:"integer"}
VALIDATION_SIZE = 0.03 # @param {type:"number"}

# Tokenization / training
CONTEXT_LENGTH = 512 # @param [128, 256, 512, 768, 1024] {type:"raw"}
PER_DEVICE_TRAIN_BATCH_SIZE = 4 # @param {type:"integer"}
PER_DEVICE_EVAL_BATCH_SIZE = 4 # @param {type:"integer"}
GRADIENT_ACCUMULATION_STEPS = 8 # @param {type:"integer"}
LEARNING_RATE = 5e-5 # @param {type:"number"}
WEIGHT_DECAY = 0.01 # @param {type:"number"}
NUM_TRAIN_EPOCHS = 1.0 # @param {type:"number"}
MAX_STEPS = 20000 # @param {type:"integer"}
WARMUP_RATIO = 0.03 # @param {type:"number"}

# Logging / checkpoints
LOGGING_STEPS = 25 # @param {type:"integer"}
EVAL_STEPS = 100 # @param {type:"integer"}
SAVE_STEPS = 100 # @param {type:"integer"}
SAVE_TOTAL_LIMIT = 2 # @param {type:"integer"}

# Generation eval
GEN_MAX_NEW_TOKENS = 140 # @param {type:"integer"}
GEN_TEMPERATURE = 0.85 # @param {type:"number"}
GEN_TOP_P = 0.92 # @param {type:"number"}
GEN_TOP_K = 50 # @param {type:"integer"}

SEED = 42 # @param {type:"integer"}
set_seed(SEED)

USE_BF16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
USE_FP16 = torch.cuda.is_available() and not USE_BF16

print("Output:", OUTPUT_DIR)
print("BF16:", USE_BF16, "| FP16:", USE_FP16)

Mounted at /content/drive
Output: /content/drive/MyDrive/gpt2_children_stories_ft/runs/gpt2_deven367_babylm-100M-children-stories
BF16: False | FP16: True


In [ ]:
# @title Prompt eval set: same prompts are used before and after fine-tuning
EVAL_PROMPTS = [
    "Once upon a time, there was a little fox who wanted to learn why the moon followed him.",
    "The children found a tiny door behind the bookshelf, and when they opened it,",
    "A brave rabbit packed a red scarf, three carrots, and a map before leaving home because",
    "Every night, the old tree whispered stories to the stars about",
    "Mira did not like going to sleep, until one evening her pillow began to",
    "The toy train under the bed started moving by itself when",
    "In a village where everyone could talk to birds, a young girl discovered",
    "The smallest dragon in the valley had one unusual problem:",
]

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
with open(f"{OUTPUT_DIR}/eval_prompts.json", "w") as f:
    json.dump(EVAL_PROMPTS, f, indent=2)

print(f"Saved {len(EVAL_PROMPTS)} eval prompts.")

Saved 8 eval prompts.


In [ ]:
# @title Load dataset
load_kwargs = {}
if DATASET_CONFIG.strip():
    load_kwargs["name"] = DATASET_CONFIG.strip()

raw_ds = load_dataset(DATASET_NAME, **load_kwargs, split=DATASET_SPLIT)
print(raw_ds)
print("Columns:", raw_ds.column_names)
print("First row:")
print(raw_ds[0])

README.md:   0%|          | 0.00/661 [00:00<?, ?B/s]

data/train-00000-of-00001-13f0a33230d64d(…):   0%|          | 0.00/10.7M [00:00<?, ?B/s]

data/valid-00000-of-00001-0177e6dbdee45e(…):   0%|          | 0.00/905k [00:00<?, ?B/s]

data/test-00000-of-00001-0b94257b623049e(…):   0%|          | 0.00/1.10M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/76758 [00:00<?, ? examples/s]

Generating valid split:   0%|          | 0/5996 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7959 [00:00<?, ? examples/s]

Dataset({
    features: ['text'],
    num_rows: 76758
})
Columns: ['text']
First row:
{'text': 'The Happy Prince.'}


In [ ]:
# @title Formatting and cleaning helpers

URL_RE = re.compile(r"https?://\S+|www\.\S+", re.IGNORECASE)
HTML_TAG_RE = re.compile(r"<[^>]+>")
SPACE_RE = re.compile(r"\s+")

def clean_text(text: Any) -> str:
    """
    Conservative cleaning for LM/SFT data.
    Avoid destructive cleaning like removing all digits or punctuation.
    """
    text = "" if text is None else str(text)
    text = URL_RE.sub(" ", text)
    text = HTML_TAG_RE.sub(" ", text)
    text = text.replace("\u00a0", " ")
    text = SPACE_RE.sub(" ", text).strip()
    return text

def pick_text_column(ds, explicit: str = "") -> str:
    if explicit.strip():
        return explicit.strip()
    preferred = ["text", "story", "content", "article", "completion", "response", "output"]
    for c in preferred:
        if c in ds.column_names:
            return c
    sample = ds[: min(100, len(ds))]
    for c in ds.column_names:
        values = sample[c]
        if any(isinstance(v, str) and len(v.strip()) > 20 for v in values):
            return c
    raise ValueError("Could not infer a text column. Set TEXT_COLUMN manually.")

def format_prompt_response(example: Dict[str, Any]) -> str:
    prompt = clean_text(example.get(PROMPT_COLUMN, ""))
    response = clean_text(example.get(RESPONSE_COLUMN, ""))

    if SYSTEM_PROMPT.strip():
        return (
            f"<|system|>\\n{clean_text(SYSTEM_PROMPT)}\\n"
            f"<|user|>\\n{prompt}\\n"
            f"<|assistant|>\\n{response}<|endoftext|>"
        )
    return f"<|user|>\\n{prompt}\\n<|assistant|>\\n{response}<|endoftext|>"

def format_raw_text(example: Dict[str, Any], text_column: str) -> str:
    return clean_text(example.get(text_column, "")) + "<|endoftext|>"

def resolve_mode() -> str:
    if DATASET_MODE != "auto":
        return DATASET_MODE
    if PROMPT_COLUMN.strip() and RESPONSE_COLUMN.strip():
        return "prompt_response"
    return "raw_text"

mode = resolve_mode()
text_column = pick_text_column(raw_ds, TEXT_COLUMN) if mode == "raw_text" else None

print("Resolved dataset mode:", mode)
if text_column:
    print("Text column:", text_column)
if mode == "prompt_response":
    if not PROMPT_COLUMN.strip() or not RESPONSE_COLUMN.strip():
        raise ValueError("For prompt_response mode, set PROMPT_COLUMN and RESPONSE_COLUMN.")
    print("Prompt column:", PROMPT_COLUMN)
    print("Response column:", RESPONSE_COLUMN)

Resolved dataset mode: raw_text
Text column: text


In [ ]:
# @title Build train/validation text dataset

def to_training_text(example):
    if mode == "prompt_response":
        text = format_prompt_response(example)
    else:
        text = format_raw_text(example, text_column)
    return {"text": text}

formatted = raw_ds.map(
    to_training_text,
    remove_columns=raw_ds.column_names,
    desc="Formatting examples",
)

formatted = formatted.filter(lambda x: len(x["text"]) > 30, desc="Dropping empty/short examples")

if MAX_TRAIN_EXAMPLES and MAX_TRAIN_EXAMPLES > 0:
    formatted = formatted.shuffle(seed=SEED)
    formatted = formatted.select(range(min(len(formatted), MAX_TRAIN_EXAMPLES + MAX_EVAL_EXAMPLES)))

split = formatted.train_test_split(test_size=VALIDATION_SIZE, seed=SEED)
train_ds = split["train"]
eval_ds = split["test"]

if MAX_EVAL_EXAMPLES and len(eval_ds) > MAX_EVAL_EXAMPLES:
    eval_ds = eval_ds.select(range(MAX_EVAL_EXAMPLES))

print(train_ds)
print(eval_ds)
print("Sample formatted text:")
print(train_ds[0]["text"][:1000])

Formatting examples:   0%|          | 0/76758 [00:00<?, ? examples/s]

Dropping empty/short examples:   0%|          | 0/76758 [00:00<?, ? examples/s]

Dataset({
    features: ['text'],
    num_rows: 20370
})
Dataset({
    features: ['text'],
    num_rows: 630
})
Sample formatted text:
The room had some resemblance to the clay-floored halls in Holstein; a pretty numerous company, consisting of seamen, Copenhagen burghers, and a few scholars, sat here in deep converse over their pewter cans, and gave little heed to the person who entered.<|endoftext|>


In [ ]:
# @title Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.config.pad_token_id = tokenizer.eos_token_id

if torch.cuda.is_available():
    model = model.to("cuda")

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Total parameters: 124,439,808
Trainable parameters: 124,439,808


In [ ]:
# @title Baseline generation before fine-tuning
def generate_many(model_or_path, tokenizer_or_path, prompts: List[str], save_path: str):
    if isinstance(model_or_path, str):
        gen_model = AutoModelForCausalLM.from_pretrained(model_or_path)
        gen_tokenizer = AutoTokenizer.from_pretrained(tokenizer_or_path)
        gen_tokenizer.pad_token = gen_tokenizer.eos_token
        if torch.cuda.is_available():
            gen_model = gen_model.to("cuda")
    else:
        gen_model = model_or_path
        gen_tokenizer = tokenizer_or_path

    gen_model.eval()
    rows = []

    for prompt in prompts:
        inputs = gen_tokenizer(prompt, return_tensors="pt").to(gen_model.device)
        with torch.no_grad():
            out = gen_model.generate(
                **inputs,
                max_new_tokens=GEN_MAX_NEW_TOKENS,
                do_sample=True,
                temperature=GEN_TEMPERATURE,
                top_p=GEN_TOP_P,
                top_k=GEN_TOP_K,
                repetition_penalty=1.08,
                pad_token_id=gen_tokenizer.eos_token_id,
                eos_token_id=gen_tokenizer.eos_token_id,
            )
        text = gen_tokenizer.decode(out[0], skip_special_tokens=True)
        rows.append({"prompt": prompt, "generated_text": text})

    with open(save_path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    return rows

baseline_path = f"{OUTPUT_DIR}/baseline_generations.jsonl"
baseline_rows = generate_many(model, tokenizer, EVAL_PROMPTS, baseline_path)

print("Saved:", baseline_path)
print("\n--- Baseline sample ---")
print(baseline_rows[0]["generated_text"])

Saved: /content/drive/MyDrive/gpt2_children_stories_ft/runs/gpt2_deven367_babylm-100M-children-stories/baseline_generations.jsonl

--- Baseline sample ---
Once upon a time, there was a little fox who wanted to learn why the moon followed him. He had no idea how it did that and he didn't know where his mother came from at all; but when she said her name again on this occasion in what is called "Apostle" (see below), I felt ashamed of myself for ever looking into my own eyes."

(See also: In other words : The Little Fox's story begins with two small children — one an orphanage worker living near Chicago named Walter Boudreau because while both were trying desperately toward understanding their parents' intentions -he saw something else too)
]

? When you look back over your life or work experience today, do you remember any moments like those seen by people not knowing about them


In [ ]:
# @title Tokenize and chunk dataset
num_proc = max(1, min(os.cpu_count() or 1, 4))

def tokenize_function(examples):
    return tokenizer(examples["text"])

tokenized_train = train_ds.map(
    tokenize_function,
    batched=True,
    num_proc=num_proc,
    remove_columns=train_ds.column_names,
    desc="Tokenizing train",
)

tokenized_eval = eval_ds.map(
    tokenize_function,
    batched=True,
    num_proc=num_proc,
    remove_columns=eval_ds.column_names,
    desc="Tokenizing eval",
)

def group_texts(examples):
    concatenated = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated["input_ids"])
    total_length = (total_length // CONTEXT_LENGTH) * CONTEXT_LENGTH

    result = {
        k: [t[i : i + CONTEXT_LENGTH] for i in range(0, total_length, CONTEXT_LENGTH)]
        for k, t in concatenated.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result

lm_train = tokenized_train.map(
    group_texts,
    batched=True,
    num_proc=num_proc,
    desc="Chunking train",
)

lm_eval = tokenized_eval.map(
    group_texts,
    batched=True,
    num_proc=num_proc,
    desc="Chunking eval",
)

print(lm_train)
print(lm_eval)

Tokenizing train (num_proc=2):   0%|          | 0/20370 [00:00<?, ? examples/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1219 > 1024). Running this sequence through the model will result in indexing errors
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1364 > 1024). Running this sequence through the model will result in indexing errors


Tokenizing eval (num_proc=2):   0%|          | 0/630 [00:00<?, ? examples/s]

Chunking train (num_proc=2):   0%|          | 0/20370 [00:00<?, ? examples/s]

Chunking eval (num_proc=2):   0%|          | 0/630 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 2272
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 69
})


In [ ]:
# @title Train GPT-2
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

args_kwargs = dict(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    max_steps=MAX_STEPS,
    warmup_ratio=WARMUP_RATIO,
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    eval_steps=EVAL_STEPS,
    save_total_limit=SAVE_TOTAL_LIMIT,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=USE_FP16,
    bf16=USE_BF16,
    report_to="none",
    dataloader_num_workers=2,
    remove_unused_columns=False,
)

sig = inspect.signature(TrainingArguments.__init__)
if "eval_strategy" in sig.parameters:
    args_kwargs["eval_strategy"] = "steps"
    args_kwargs["save_strategy"] = "steps"
    args_kwargs["logging_strategy"] = "steps"
else:
    args_kwargs["evaluation_strategy"] = "steps"
    args_kwargs["save_strategy"] = "steps"
    args_kwargs["logging_strategy"] = "steps"

training_args = TrainingArguments(**args_kwargs)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=lm_train,
    eval_dataset=lm_eval,
    data_collator=data_collator,
)

train_result = trainer.train()
metrics = trainer.evaluate()

print("Train result:", train_result)
print("Eval metrics:", metrics)

Path(FINAL_MODEL_DIR).mkdir(parents=True, exist_ok=True)
trainer.save_model(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)

with open(f"{OUTPUT_DIR}/train_metrics.json", "w") as f:
    json.dump(train_result.metrics, f, indent=2)

with open(f"{OUTPUT_DIR}/eval_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("Saved final model:", FINAL_MODEL_DIR)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss,Validation Loss
100,3.828368,3.666231
200,3.700111,3.576247
300,3.615045,3.534167
400,3.557240,3.507759
500,3.505506,3.486580
600,3.433342,3.472092
700,3.386576,3.459863
800,3.310950,3.450488


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
# @title Fine-tuned generation on the same eval prompts
torch.cuda.empty_cache()

ft_model = AutoModelForCausalLM.from_pretrained(FINAL_MODEL_DIR)
ft_tokenizer = AutoTokenizer.from_pretrained(FINAL_MODEL_DIR)
ft_tokenizer.pad_token = ft_tokenizer.eos_token
ft_model.config.pad_token_id = ft_tokenizer.eos_token_id

if torch.cuda.is_available():
    ft_model = ft_model.to("cuda")

finetuned_path = f"{OUTPUT_DIR}/finetuned_generations.jsonl"
finetuned_rows = generate_many(ft_model, ft_tokenizer, EVAL_PROMPTS, finetuned_path)

print("Saved:", finetuned_path)
print("\n--- Fine-tuned sample ---")
print(finetuned_rows[0]["generated_text"])

In [ ]:
# @title Save before/after comparison
comparison_rows = []
for base, ft in zip(baseline_rows, finetuned_rows):
    comparison_rows.append({
        "prompt": base["prompt"],
        "baseline": base["generated_text"],
        "finetuned": ft["generated_text"],
    })

comparison_json = f"{OUTPUT_DIR}/generation_comparison.json"
with open(comparison_json, "w", encoding="utf-8") as f:
    json.dump(comparison_rows, f, indent=2, ensure_ascii=False)

comparison_md = f"{OUTPUT_DIR}/generation_comparison.md"
with open(comparison_md, "w", encoding="utf-8") as f:
    f.write("# GPT-2 Before vs After Fine-Tuning\n\n")
    f.write(f"Dataset: `{DATASET_NAME}`\n\n")
    f.write(f"Base model: `{MODEL_NAME}`\n\n")
    for i, row in enumerate(comparison_rows, 1):
        f.write(f"## Prompt {i}\n\n")
        f.write(f"**Prompt**\n\n{row['prompt']}\n\n")
        f.write("**Before fine-tuning**\n\n")
        f.write(row["baseline"] + "\n\n")
        f.write("**After fine-tuning**\n\n")
        f.write(row["finetuned"] + "\n\n")
        f.write("---\n\n")

print("Saved:", comparison_json)
print("Saved:", comparison_md)

In [ ]:
# @title Optional: zip final model and outputs
import shutil

zip_base = f"{OUTPUT_DIR}/gpt2_finetuned_artifacts"
zip_path = shutil.make_archive(zip_base, "zip", OUTPUT_DIR)
print("Zipped:", zip_path)

Zipped: /content/drive/MyDrive/gpt2_children_stories_ft/runs/gpt2_deven367_babylm-100M-children-stories/gpt2_finetuned_artifacts.zip


In [ ]:
# @title Optional: push model to Hugging Face Hub
PUSH_TO_HUB = False # @param {type:"boolean"}
HF_REPO_ID = "" # @param {type:"string"}

if PUSH_TO_HUB:
    if not HF_TOKEN:
        raise ValueError("Set HF_TOKEN in Colab Secrets before pushing to the Hub.")
    if not HF_REPO_ID.strip():
        raise ValueError("Set HF_REPO_ID, e.g. username/gpt2-children-stories.")
    ft_model.push_to_hub(HF_REPO_ID)
    ft_tokenizer.push_to_hub(HF_REPO_ID)
    print("Pushed to:", HF_REPO_ID)
else:
    print("Skipping push_to_hub.")

Skipping push_to_hub.


In [ ]:
# @title Interactive inference
prompt = "Once upon a time, a small cat discovered" # @param {type:"string"}
max_new_tokens = 240 # @param {type:"integer"}
temperature = 0.85 # @param {type:"number"}
top_p = 0.92 # @param {type:"number"}

inputs = ft_tokenizer(prompt, return_tensors="pt").to(ft_model.device)
with torch.no_grad():
    outputs = ft_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        repetition_penalty=1.08,
        pad_token_id=ft_tokenizer.eos_token_id,
        eos_token_id=ft_tokenizer.eos_token_id,
    )

print(ft_tokenizer.decode(outputs[0], skip_special_tokens=True))

Once upon a time, a small cat discovered that there was an old man living in the village who lived very near. He had his horse and ran away with it; but as soon as he came out of his home on foot, everything went to hell for him! The little cats jumped up from their shelter-bearers' feet without ever saying one word about this strange creature or what they heard inside. When everyone looked at each other, though, another Cat sprang into view: "Oh my lord!" cried she, "what are you doing here? What do you have come hither?" And after running along alone all night long, her whole body seemed so happy again! All morning even while lying down by himself under great weight, these wonderful things were eating through its flesh like worms." A short distance further than usual began to get worse before finally stopping altogether until someone stopped them outside—a tall woman standing over herself beside something else whose mouth opened once more around itself which said nothing good except 

## Notes

- The old `TextDataset` API is avoided; this notebook uses `datasets.map()` + chunking.
- For story continuation, raw causal language modeling is correct.
- For actual SFT/instruction datasets, set:
  - `DATASET_MODE = "prompt_response"`
  - `PROMPT_COLUMN = "..."`
  - `RESPONSE_COLUMN = "..."`
- Keep the same prompt eval set fixed. Do not tune prompts between before/after runs.
- If training is too slow:
  - set `MODEL_NAME = "distilgpt2"`
  - reduce `MAX_TRAIN_EXAMPLES`
  - reduce `CONTEXT_LENGTH`
  - increase `GRADIENT_ACCUMULATION_STEPS` instead of batch size